## 1) Setup and Imports

This cell imports Python libraries used for file handling, date parsing, and data cleaning with pandas.

In [7]:
# Core utilities for path handling, parsing, and tabular data processing.
from pathlib import Path
import json
import re
from datetime import datetime

import pandas as pd
import numpy as np

print("Libraries imported")

Libraries imported


## 2) Input Configuration and File Discovery

This cell defines where the Excel files are located, filters temporary Excel files, and previews what was found.

In [8]:
# Project-level settings used throughout the notebook.
INPUT_ROOT = Path("./data")
SHEET_NAME = "Main Prod"
HEADER_ROW = 11
OUTPUT_CSV = Path("combined_daily_reports.csv")

# Discover all Excel reports and ignore temporary lock files (~$...).
files = sorted(INPUT_ROOT.rglob("*.xlsx"))
files = [p for p in files if not p.name.startswith("~$")]

print("Input root:", INPUT_ROOT.resolve())
print("Files found:", len(files))
print("First files:", [p.name for p in files[:5]])

Input root: C:\Users\OPS010\Documents\projects\new QC\data
Files found: 1092
First files: ['01-01-2022.xlsx', '02-01-2022.xlsx', '03-01-2022.xlsx', '04-01-2022.xlsx', '05-01-2022.xlsx']


## 3) Column Rules and Helper Functions

This cell defines required output columns, keyword matching rules, and helper functions for text normalization, date parsing, and automatic column mapping.

In [9]:
# Canonical keys expected in the cleaned output.
REQUIRED_KEYS = ["field", "station", "capacity", "oil_prod", "actual", "gravity", "gas_prod", "bsw", "water"]
OUTPUT_RENAME = {
    "field": "field",
    "station": "station",
    "capacity": "capacity",
    "oil_prod": "oil prod",
    "actual": "actual",
    "gravity": "gravity",
    "gas_prod": "gas prod",
    "bsw": "bs&w",
    "water": "water",
}

# Keyword groups to help auto-detect similar column names across files.
KEYWORDS = {
    "field": ["field"],
    "station": ["station", "stn"],
    "oil_prod": ["oil", "oil prod", "oil production", "bopd", "bbl"],
    "gas_prod": ["gas", "gas prod", "gas production", "mmscf", "scf"],
    "water": ["water", "water prod", "water production"],
    "bsw": ["bs&w", "bsw", "bs and w", "bs & w"],
}

def normalize(text):
    # Normalize labels for robust case/spacing/punctuation-insensitive matching.
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def parse_date_from_filename(name):
    # Extract date token from filename and parse common DD-MM-YYYY / MM-DD-YYYY variants.
    match = re.search(r"(\d{1,2}[-_ ]\d{1,2}[-_ ]\d{2,4})", name)
    if not match:
        raise ValueError(f"No date found in filename: {name}")
    raw = match.group(1).replace("_", "-").replace(" ", "-")
    for fmt in ("%d-%m-%Y", "%d-%m-%y", "%m-%d-%Y", "%m-%d-%y"):
        try:
            return datetime.strptime(raw, fmt).date()
        except ValueError:
            continue
    raise ValueError(f"Unrecognized date format in filename: {name}")


def auto_map_columns(columns):
    # Try to map each canonical key to one detected source column.
    norm_cols = {c: normalize(c) for c in columns}
    mapping = {}
    for key, words in KEYWORDS.items():
        candidates = []
        for col, norm in norm_cols.items():
            if any(word in norm for word in words):
                candidates.append(col)
        mapping[key] = candidates[0] if len(candidates) == 1 else None
    return mapping

## 4) Inspect a Sample Report and Set Column Map

This cell reads one sample Excel report to inspect column names and sets a manual column mapping for fields used later.

In [10]:
# Inspect one file to confirm exact source headers before full processing.
sample = pd.read_excel(files[0], sheet_name=SHEET_NAME, header=HEADER_ROW, usecols="A:I")
sample.columns = sample.columns.map(lambda value: str(value).strip())
print("Sample columns:", list(sample.columns))

# Manual map from canonical keys to report-specific column labels.
column_map = {
    "field": "CONC.",
    "station": "STATIONS",
    "oil_prod": ["OIL", "EXPORT"],
    "actual": ["ACTUAL"],
    "gas_prod": ["GAS", "PRODUCED"],
    "water": "WATER",
}
print("Column map:", column_map)

Sample columns: ['CONC.', 'STATIONS', 'CAPACITY', 'Unnamed: 3', 'OIL', 'GRAVITY', 'BS&W', 'WATER', 'GAS']
Column map: {'field': 'CONC.', 'station': 'STATIONS', 'oil_prod': ['OIL', 'EXPORT'], 'actual': ['ACTUAL'], 'gas_prod': ['GAS', 'PRODUCED'], 'water': 'WATER'}


## 5) Process All Files, Clean Data, and Export CSV

This cell loops through all reports, resolves source columns, normalizes values, removes totals/noise rows, and writes the combined cleaned dataset to CSV.

In [11]:
# Fill column_map manually here if the report headers ever change.
missing = [key for key, value in column_map.items() if value is None and key not in {"actual", "gas_prod"}]
if missing:
    raise ValueError("Missing column mapping for: " + ", ".join(missing))


def pick_column(columns, candidates):
    # Resolve the first candidate that matches an actual column name.
    normalized = {normalize(column): column for column in columns}
    for candidate in candidates:
        resolved = normalized.get(normalize(candidate))
        if resolved is not None:
            return resolved
    return None


def normalize_field_name(value):
    # Standardize field labels and drop rows that are not real field entries.
    text = normalize(value)
    if not text or "pipe line" in text or "pipeline" in text:
        return pd.NA
    if "gialo" in text:
        return "GIALO"
    if "waha" in text:
        return "WAHA"
    if "dahra" in text:
        return "DAHRA"
    if "samah" in text:
        return "SAMAH"
    return pd.NA


records = []
for path in files:
    # Read each daily report and normalize raw headers.
    df = pd.read_excel(path, sheet_name=SHEET_NAME, header=HEADER_ROW, usecols="A:I")
    df.columns = df.columns.map(lambda value: str(value).strip())
    report_date = parse_date_from_filename(path.name)

    # Resolve source columns for each required output field.
    field_source = pick_column(df.columns, [column_map["field"]])
    station_source = pick_column(df.columns, [column_map["station"]])
    capacity_source = pick_column(df.columns, ["CAPACITY"])
    oil_source = pick_column(df.columns, column_map["oil_prod"])
    actual_source = pick_column(df.columns, column_map["actual"])
    gas_source = pick_column(df.columns, column_map["gas_prod"])
    bsw_source = pick_column(df.columns, ["BS&W", "BS & W", "BSW"])
    gravity_source = pick_column(df.columns, ["GRAVITY", "API"])
    water_source = pick_column(df.columns, [column_map["water"]])

    # Fail fast if mandatory sources are missing in a file.
    missing_sources = [
        name
        for name, source in {
            "field": field_source,
            "station": station_source,
            "capacity": capacity_source,
            "oil_prod": oil_source,
            "gravity": gravity_source,
            "bsw": bsw_source,
            "water": water_source,
        }.items()
        if source is None
    ]
    if missing_sources:
        raise ValueError(f"Missing source columns in {path.name}: {', '.join(missing_sources)}")

    # Build canonical columns used by the final dataset.
    df = df.copy()
    df["field"] = df[field_source].ffill()
    df["station"] = df[station_source]
    df["capacity"] = df[capacity_source]
    df["oil_prod"] = df[oil_source]
    df["actual"] = df[actual_source] if actual_source is not None else df["oil_prod"]
    df["gas_prod"] = df[gas_source] if gas_source is not None else pd.NA
    df["gravity"] = df[gravity_source]
    df["bsw"] = df[bsw_source]
    df["water"] = df[water_source]

    # Clean text columns and standardize field values.
    for text_col in ["field", "station"]:
        df[text_col] = df[text_col].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    df["field"] = df["field"].map(normalize_field_name)

    # Remove summary/total rows from either field or station labels.
    total_mask = (
        df["field"].astype(str).str.contains("total", case=False, na=False)
        | df["station"].astype(str).str.contains("total", case=False, na=False)
    )

    output_columns = ["field", "station", "capacity", "oil_prod", "actual", "gravity", "bsw", "gas_prod", "water"]
    df = df.loc[df["field"].notna() & df["station"].notna() & ~total_mask, output_columns].copy()
    df.insert(0, "date", report_date)

    # Coerce numeric columns and backfill missing actual from oil production.
    numeric_columns = ["capacity", "oil_prod", "actual", "gas_prod", "gravity", "bsw", "water"]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df["actual"] = df["actual"].fillna(df["oil_prod"])
    df = df.dropna(how="all", subset=output_columns)
    records.append(df)

# Merge all cleaned reports and export final CSV.
combined = pd.concat(records, ignore_index=True)
combined = combined.rename(columns=OUTPUT_RENAME)
combined.to_csv(OUTPUT_CSV, index=False)
print("Wrote", len(combined), "rows to", OUTPUT_CSV.resolve())
combined.head()

Wrote 19241 rows to C:\Users\OPS010\Documents\projects\new QC\combined_daily_reports.csv


,date,field,station,capacity,oil prod,actual,gravity,bs&w,gas prod,water
0,2022-01-01,GIALO,GIALO-1,40000.0,38118,38118.0,35.9,0.10,0.000,238875.0
1,2022-01-01,GIALO,GIALO-2,59000.0,53507,53507.0,37.5,0.10,8.016,172343.0
2,2022-01-01,GIALO,MASRAB,8500.0,6898,6898.0,41.6,0.05,4.606,90.0
3,2022-01-01,GIALO,6K AREA,NaN,0,0.0,0.0,0.00,0.000,0.0
4,2022-01-01,GIALO,3V AREA,NaN,0,0.0,0.0,0.00,0.000,0.0


In [12]:
# --- Debug: List files missing the expected sheet ---
# import openpyxl
# from pandas import ExcelFile
# missing_sheet = []
# sheet_names = {}
# for path in files:
#     try:
#         with ExcelFile(path) as xls:
#             names = xls.sheet_names
#             sheet_names[path.name] = names
#             if SHEET_NAME not in names:
#                 missing_sheet.append(path.name)
#     except Exception as e:
#         print(f"Error reading {path.name}: {e}")
# print("\nFiles missing the expected sheet:")
# for fname in missing_sheet:
#     print(f"- {fname} (sheets: {sheet_names.get(fname)})")
# print(f"\nTotal missing: {len(missing_sheet)} out of {len(files)} files.")